# Cross-Party Mentions and Sentiment

This notebook analyzes when ads from one main party mention other parties or leaders, and compares lexicon-based and transformer-based sentiment on those mentions.

In [ ]:
%load_ext watermark
%watermark -v -n -m -p numpy,pandas,matplotlib,seaborn,sklearn,nltk,transformers,torch

import os
import sys
import re
import warnings
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings('ignore')

%load_ext autoreload
%autoreload 2
%matplotlib inline

print(f'default sys.path: {sys.path}')
PROJ_ROOT = os.path.abspath(os.path.join(os.pardir))
PROJ_ROOT = os.path.abspath(os.path.join(PROJ_ROOT, os.pardir))
sys.path.append(PROJ_ROOT)
print(f'Project root: {PROJ_ROOT}')

from utils import mpl_settings
from utils.dataset_utilities import load_data

mpl_settings.apply_settings()
plt.style.use(mpl_settings.plot_settings)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
file_path = '../../data/processed/2022_aus_elections_mar_to_may.csv'
df = load_data(file_path)

main_parties = ['Liberal', 'Labor', 'Green', 'Independent', 'United Australia Party']
df = df[df['macro_party_uap'].isin(main_parties)].copy()
df = df[df['ad_creative_body'].notna()].copy()

print(f'Rows after main-party filter: {df.shape[0]:,}')
print(df['macro_party_uap'].value_counts().to_string())


## Mention dictionaries

We match both party names and leader names. The patterns are intentionally permissive so that variants such as Labor/Labour and UAP/United Australia Party are captured.

In [ ]:
def normalize_text(text):
    if pd.isna(text):
        return ''
    return re.sub(r'\s+', ' ', str(text)).strip().lower()

party_aliases = {
    'Labor': [
        r'\blabor\b',
        r'\blabour\b',
        r'\baustralian labor party\b',
        r'\balp\b',
        r'\banthony albanese\b',
        r'\balbanese\b',
    ],
    'Liberal': [
        r'\bliberal\b',
        r'\bliberal party\b',
        r'\bliberal nationals?\b',
        r'\bthe coalition\b',
        r'\bscott morrison\b',
        r'\bmorrison\b',
        r'\bcoalition\b',
    ],
    'Green': [
        r'\bgreen\b',
        r'\bgreens\b',
        r'\baustralian greens\b',
        r'\badam bandt\b',
        r'\bbandt\b',
    ],
    'United Australia Party': [
        r'\bunited australia party\b',
        r'\buap\b',
        r'\bclive palmer\b',
        r'\bpalmer\b',
    ],
    'Independent': [
        r'\bindependent\b',
        r'\bteal\b',
        r'\bteals\b',
    ],
}

compiled_aliases = {
    party: [re.compile(pattern, flags=re.IGNORECASE) for pattern in patterns]
    for party, patterns in party_aliases.items()
}

text_columns = [c for c in ['ad_creative_body', 'ad_creative_link_title', 'ad_creative_link_description', 'ad_creative_link_caption'] if c in df.columns]

def build_text(row):
    parts = []
    for col in text_columns:
        value = row.get(col)
        if pd.notna(value) and str(value).strip():
            parts.append(str(value).strip())
    return ' '.join(parts)

df['analysis_text'] = df.apply(build_text, axis=1)

def detect_mentions(text):
    text = normalize_text(text)
    mentions = []
    for target_party, patterns in compiled_aliases.items():
        for pattern in patterns:
            if pattern.search(text):
                mentions.append(target_party)
                break
    return sorted(set(mentions))

df['mentioned_parties'] = df['analysis_text'].apply(detect_mentions)

def cross_mentions_only(row):
    source = row['macro_party_uap']
    return [target for target in row['mentioned_parties'] if target != source]

df['cross_party_mentions'] = df.apply(cross_mentions_only, axis=1)
df['has_cross_party_mention'] = df['cross_party_mentions'].apply(bool)

print('Ads with at least one mention of another main party or leader:')
print(df['has_cross_party_mention'].mean())


In [ ]:
# Long-form cross-party mention dataframe
mention_rows = []
for _, row in df.iterrows():
    source_party = row['macro_party_uap']
    for target_party in row['cross_party_mentions']:
        mention_rows.append({
            'id': row['id'],
            'source_party': source_party,
            'target_party': target_party,
            'analysis_text': row['analysis_text'],
            'ad_creative_body': row.get('ad_creative_body', ''),
            'ad_creative_link_title': row.get('ad_creative_link_title', ''),
            'ad_creative_link_description': row.get('ad_creative_link_description', ''),
            'ad_creative_link_caption': row.get('ad_creative_link_caption', ''),
            'mentioned_parties': row['mentioned_parties'],
        })

mention_columns = [
    'id', 'source_party', 'target_party', 'analysis_text',
    'ad_creative_body', 'ad_creative_link_title', 'ad_creative_link_description',
    'ad_creative_link_caption', 'mentioned_parties'
]
mentions_df = pd.DataFrame(mention_rows, columns=mention_columns)
print(f'Long-form mention rows: {mentions_df.shape[0]:,}')
mentions_df.head()


In [ ]:
# Counts matrix and mention rates
party_order = main_parties
counts_matrix = pd.crosstab(mentions_df['source_party'], mentions_df['target_party'])
counts_matrix = counts_matrix.reindex(index=party_order, columns=party_order, fill_value=0)

source_totals = df.groupby('macro_party_uap')['id'].count().reindex(party_order)
mention_totals = mentions_df.groupby('source_party')['id'].nunique().reindex(party_order).fillna(0)
mention_rates = (mention_totals / source_totals).replace([np.inf, np.nan], 0)

print('Cross-party mention counts matrix:')
print(counts_matrix)
print()
print('Mention rates by source party:')
print(mention_rates.to_string())


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(counts_matrix, annot=True, fmt='d', cmap='Blues', ax=ax, cbar_kws={'label': 'Mention count'})
ax.set_title('Cross-party mentions: source party vs target party')
ax.set_xlabel('Target party mentioned')
ax.set_ylabel('Source party')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
mention_rates.sort_values(ascending=False).plot(kind='bar', color='steelblue', ax=ax)
ax.set_ylabel('Share of ads mentioning another party')
ax.set_xlabel('Source party')
ax.set_title('Mention rate by source party')
ax.set_ylim(0, max(mention_rates.max() * 1.15, 0.05))
plt.tight_layout()
plt.show()


## Sentiment analysis

Two sentiment methods are used: VADER and a transformer pipeline. If a package or model is unavailable, the notebook falls back gracefully and keeps running.

In [ ]:
# Lexicon-based sentiment: VADER
vader_available = False
try:
    import nltk
    from nltk.sentiment import SentimentIntensityAnalyzer
    try:
        nltk.data.find('sentiment/vader_lexicon.zip')
    except LookupError:
        nltk.download('vader_lexicon', quiet=True)
    vader = SentimentIntensityAnalyzer()
    vader_available = True
    print('VADER loaded successfully.')
except Exception as exc:
    vader = None
    print(f'VADER unavailable: {exc}')

# Transformer-based sentiment
transformer_available = False
transformer_pipe = None
try:
    from transformers import pipeline
    transformer_pipe = pipeline(
        'sentiment-analysis',
        model='distilbert-base-uncased-finetuned-sst-2-english',
        truncation=True,
    )
    transformer_available = True
    print('Transformer sentiment pipeline loaded successfully.')
except Exception as exc:
    print(f'Transformer sentiment unavailable: {exc}')


def vader_score(text):
    if not vader_available:
        return np.nan
    return vader.polarity_scores(text)['compound']


def transformer_score(text):
    if not transformer_available:
        return np.nan
    try:
        result = transformer_pipe(text[:512])[0]
        score = float(result['score'])
        return score if result['label'].upper().startswith('POS') else -score
    except Exception:
        return np.nan

# compute per-ad sentiment on the analysis text
sample_text = df['analysis_text'].fillna('').astype(str)
df['vader_sentiment'] = sample_text.apply(vader_score)
df['transformer_sentiment'] = sample_text.apply(transformer_score)

print(df[['vader_sentiment', 'transformer_sentiment']].describe(include='all'))


In [ ]:
# Attach sentiment scores to long-form mention rows
def row_sentiment_lookup(row):
    return pd.Series({
        'vader_sentiment': row['vader_sentiment'],
        'transformer_sentiment': row['transformer_sentiment'],
    })

mentions_df = mentions_df.merge(
    df[['id', 'vader_sentiment', 'transformer_sentiment']],
    on='id',
    how='left'
)

sentiment_summary = mentions_df.groupby(['source_party', 'target_party']).agg(
    mention_count=('id', 'count'),
    vader_mean=('vader_sentiment', 'mean'),
    vader_median=('vader_sentiment', 'median'),
    transformer_mean=('transformer_sentiment', 'mean'),
    transformer_median=('transformer_sentiment', 'median'),
).reset_index()

print(sentiment_summary.head(20).to_string(index=False))


In [ ]:
# Sentiment summaries and visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

if vader_available:
    vader_plot = sentiment_summary.pivot(index='source_party', columns='target_party', values='vader_mean').reindex(index=party_order, columns=party_order)
    sns.heatmap(vader_plot, annot=True, cmap='RdYlGn', center=0, ax=axes[0], cbar_kws={'label': 'Mean VADER compound'})
    axes[0].set_title('Mean VADER sentiment by source -> target')
else:
    axes[0].axis('off')
    axes[0].set_title('VADER unavailable')

if transformer_available:
    transformer_plot = sentiment_summary.pivot(index='source_party', columns='target_party', values='transformer_mean').reindex(index=party_order, columns=party_order)
    sns.heatmap(transformer_plot, annot=True, cmap='RdYlGn', center=0, ax=axes[1], cbar_kws={'label': 'Mean transformer sentiment'})
    axes[1].set_title('Mean transformer sentiment by source -> target')
else:
    axes[1].axis('off')
    axes[1].set_title('Transformer unavailable')

plt.tight_layout()
plt.show()

if mentions_df.shape[0] > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    plot_df = mentions_df.copy()
    plot_df['pair'] = plot_df['source_party'] + ' -> ' + plot_df['target_party']
    top_pairs = plot_df['pair'].value_counts().head(12).index
    sns.boxplot(data=plot_df[plot_df['pair'].isin(top_pairs)], x='pair', y='vader_sentiment', ax=ax, color='lightsteelblue')
    ax.set_title('VADER sentiment distribution for top cross-party mention pairs')
    ax.set_xlabel('Source -> target pair')
    ax.set_ylabel('VADER compound score')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()


## Sample ads

The following tables show examples of ads mentioning selected target parties, together with sentiment scores.

In [ ]:
def sample_ads_for_pair(source_party, target_party, n=5):
    subset = mentions_df[(mentions_df['source_party'] == source_party) & (mentions_df['target_party'] == target_party)].copy()
    if subset.empty:
        return subset
    subset = subset.merge(
        df[['id', 'ad_creative_body', 'analysis_text', 'vader_sentiment', 'transformer_sentiment']],
        on='id',
        how='left',
        suffixes=('', '_ad')
    )
    subset = subset[['id', 'source_party', 'target_party', 'vader_sentiment', 'transformer_sentiment', 'ad_creative_body', 'analysis_text']]
    subset = subset.drop_duplicates(subset=['id']).sample(min(n, subset.shape[0]), random_state=RANDOM_STATE)
    return subset

selected_pairs = [
    ('Liberal', 'Labor'),
    ('Labor', 'Liberal'),
    ('Green', 'Labor'),
    ('Independent', 'Liberal'),
    ('United Australia Party', 'Labor'),
]

for source_party, target_party in selected_pairs:
    sample = sample_ads_for_pair(source_party, target_party, n=5)
    print()
    print(f'### {source_party} -> {target_party}')
    if sample.empty:
        print('No matches found.')
    else:
        display(sample.reset_index(drop=True))


## Notes

- The notebook keeps only the five main parties throughout.
- Mention detection is regex-based and intentionally broad; if you want stricter matching, the alias list can be tightened.
- The transformer step loads a standard SST-2 model and falls back cleanly if the environment cannot fetch or run it.